# Load & Clean CL scotus data

# Import libraries

In [ ]:
import numpy as np
import pandas as pd

from cl_utils import *
from clean_utils import *

# Load the CL scotus data, to prepare for joining with scotus scraped data

In [ ]:
scotus_df = load_courts_to_df(["scotus"], "cl_scotus")
scotus_df.head()

,docket_id,docket_number_raw,cluster_id,case_name,date_filed,precedential_status,court_id,docket_type,docket_number,docket_format_confirmed
0,19857,82-7235,418659,United States v. Cunningham,1983-05-25,Published,ca11,singles,82-7235,True
1,6062,16-13336,2645611,Cary Moore Menzie v. Ann Taylor Retail Inc.,2013-12-11,Unpublished,ca11,singles,16-13336,True
2,34806,04-11671; D.C. Docket 02-00172-CR-1-1,41842,United States v. Ernest Clifford Miller,2005-12-02,Unpublished,ca11,singles,04-11671,True
3,6114,17-15566,2645610,Michael D. Hilderbrandt v. L.T. Butts,2013-12-11,Unpublished,ca11,singles,17-15566,True
4,1489,19-11292,2646758,"Allen James Starks v. Warden, FCC Coleman-USP I",2013-12-19,Unpublished,ca11,singles,19-11292,True


In [ ]:
len(scotus_df)

1441430

In [ ]:
scotus_df["court_id"].value_counts()

court_id
ca9     235091
ca5     204099
ca4     181253
ca2     126230
ca6     110257
ca8     108531
ca3     104716
ca11     99517
ca7      96533
ca10     74265
cadc     54571
ca1      46367
Name: count, dtype: int64

In [ ]:
scotus_df["docket_type"].value_counts()

docket_type
singles                 1214207
multiples                150559
simple_num_ranges         35643
simple_format_ranges      18156
other                     15145
thousands                  6833
slashes                     471
delimiteds                  416
Name: count, dtype: int64

In [ ]:
scotus_df["docket_format_confirmed"].value_counts()

docket_format_confirmed
True     1426283
False      15147
Name: count, dtype: int64

In [ ]:
scotus_df[["docket_type", "docket_format_confirmed"]].value_counts()

docket_type           docket_format_confirmed
singles               True                       1214205
multiples             True                        150559
simple_num_ranges     True                         35643
simple_format_ranges  True                         18156
other                 False                        15145
thousands             True                          6833
slashes               True                           471
delimiteds            True                           416
singles               False                            2
Name: count, dtype: int64

In [ ]:
len(scotus_df[scotus_df["docket_type"] != "other"])

1426285

In [ ]:
len(scotus_df[scotus_df["docket_type"] != "other"][["docket_number_raw", "court_id"]].drop_duplicates())

1154431

In [ ]:
len(scotus_df[scotus_df["docket_type"] != "other"][["docket_number", "court_id"]].drop_duplicates())

1159987

## Extract the records for manual review

In [9]:
review_df = pd.read_csv("cl_circuits.csv")
review_df["docket_number_raw"].isna().sum()

np.int64(0)

In [19]:
len(review_df)

1169291

In [10]:
review_df = review_df[["docket_number_raw", "court_id"]].dropna().drop_duplicates()

review_df[["docket_type", "docket_number"]] = review_df.apply(
        lambda row: pd.Series(clean_docket_numbers(row["docket_number_raw"])), axis=1
    )

review_df.head()

,docket_number_raw,court_id,docket_type,docket_number
0,82-7235,ca11,singles,[82-7235]
1,16-13336,ca11,singles,[16-13336]
2,04-11671; D.C. Docket 02-00172-CR-1-1,ca11,singles,[04-11671]
3,17-15566,ca11,singles,[17-15566]
4,19-11292,ca11,singles,[19-11292]


In [11]:
review_df["docket_format_confirmed"] = review_df["docket_number"].apply(
    lambda dockets: [confirm_docket_format(d) for d in dockets]
)

review_df.head()

,docket_number_raw,court_id,docket_type,docket_number,docket_format_confirmed
0,82-7235,ca11,singles,[82-7235],[True]
1,16-13336,ca11,singles,[16-13336],[True]
2,04-11671; D.C. Docket 02-00172-CR-1-1,ca11,singles,[04-11671],[True]
3,17-15566,ca11,singles,[17-15566],[True]
4,19-11292,ca11,singles,[19-11292],[True]


In [12]:
review_df["docket_type"].value_counts()

docket_type
singles                 1084038
multiples                 58414
other                     14860
thousands                  6334
simple_num_ranges          3110
simple_format_ranges       2196
slashes                     210
delimiteds                  129
Name: count, dtype: int64

In [14]:
len(review_df[review_df["docket_type"] == "other"])

14860

In [18]:
len(review_df[review_df["docket_type"] != "other"])

1154431

In [15]:
review_df.to_csv("manual_docket_reviews.csv", index=False)